# 05 — H1 Strengthening

Three literature-motivated additions to harden the H1 (Agenda Distortion) findings:

1. **Global model as parallel validation** — Train a single BERTopic on all 20k docs, compare JSD rankings to merged model. Validates the merged-model approach against the literature standard (Muller & Freudenthaler 2022; Jacobi et al. 2016).

2. **Outlet clustering + dendrogram** — Pairwise JSD matrix → hierarchical clustering (Ward's method). Tests whether outlets cluster by ideological category (Muller & Freudenthaler 2022).

3. **Chi-squared per-topic significance** — For each (outlet, topic) pair, test whether the observed topic share differs significantly from Tagesschau. Turns raw percentage differences into statistically grounded claims (FDR-corrected).

In [ ]:
import sys
from pathlib import Path

# Project paths
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"

sys.path.insert(0, str(NOTEBOOK_DIR))
sys.path.insert(0, str(PROJECT_ROOT / "1a_BERTopic"))

from modeling import load_iteration, IterationParams
from metrics import (
    compute_all_h1_metrics,
    chi_squared_topic_tests,
    summarize_significant_topics,
    pairwise_outlet_jsd,
)
from visualization import (
    plot_pairwise_jsd_heatmap,
    plot_outlet_dendrogram,
    plot_chi_squared_volcano,
)
from merged_outlets_analysis import THESIS_COLORS

# Load existing v1 iteration (merged model results)
v1 = load_iteration(OUTPUT_DIR, "v1")
print(f"Loaded v1: {v1.n_topics} topics, {len(v1.merged_articles):,} articles")

---
## 1. Outlet Clustering & Dendrogram

Compute pairwise JSD between all 7 outlets and cluster them using Ward's method.

**Question**: Do outlets cluster by their a priori ideological category (Pro-Russian, Right-Wing, Right-Populist)?

**Literature**: Muller & Freudenthaler (2022) found two clusters in 9 German alt outlets — core right-wing populist vs topically diverse.

In [ ]:
# Pairwise JSD matrix
jsd_matrix = pairwise_outlet_jsd(v1.merged_articles)
print("Pairwise JSD matrix:")
print(jsd_matrix.round(3).to_string())

# Save
fig_dir = OUTPUT_DIR / "v1" / "figures"
jsd_matrix.to_csv(OUTPUT_DIR / "v1" / "pairwise_jsd.csv")

In [ ]:
# Pairwise JSD heatmap (cluster-ordered)
fig_heatmap = plot_pairwise_jsd_heatmap(
    jsd_matrix,
    colors=THESIS_COLORS,
    save_path=fig_dir / "h1_08_pairwise_jsd_heatmap.pdf",
)

In [ ]:
# Dendrogram (Ward's method)
fig_dendro = plot_outlet_dendrogram(
    jsd_matrix,
    colors=THESIS_COLORS,
    save_path=fig_dir / "h1_09_outlet_dendrogram.pdf",
)

**Interpretation**: Check the dendrogram for:
- Do Pro-Russian outlets (RT, Antispiegel) cluster together?
- Do Right-Wing outlets (Compact, Deutschlandkurier) cluster together?
- Does Tagesschau stand alone as the most distant from all alt outlets?
- Where do Right-Populist outlets (Nius, Tichys Einblick) fall?

If clustering matches ideological categories, this validates the outlet categorization empirically.

---
## 2. Chi-Squared Per-Topic Significance Tests

For each (outlet, topic) pair, test whether the observed topic proportion differs
significantly from Tagesschau using chi-squared tests with FDR correction (Benjamini-Hochberg).

This turns raw claims like "DK has 12% AfD coverage vs Tagesschau's 2.3%" into
statistically grounded statements with effect sizes and corrected p-values.

In [ ]:
# Run chi-squared tests for all (outlet, topic) pairs
chi_df = chi_squared_topic_tests(v1.merged_articles, reference_label="Tagesschau")
print(f"Total tests: {len(chi_df)}")
print(f"Significant (FDR < 0.01): {chi_df['significant'].sum()}")
print(f"Significant with |diff| >= 1pp: {((chi_df['significant']) & (chi_df['diff_pp'].abs() >= 1.0)).sum()}")

# Save full results
chi_df.to_csv(OUTPUT_DIR / "v1" / "chi_squared_topic_tests.csv", index=False)

In [ ]:
# Summarize: significant topics with >= 1pp difference
sig_topics = summarize_significant_topics(chi_df, alpha=0.01, min_diff_pp=1.0)
print(f"\n{'='*80}")
print(f"Significant over/under-representations (p_corrected < 0.01, |diff| >= 1pp)")
print(f"{'='*80}\n")

# Show top over-representations per outlet
for outlet in sorted(sig_topics["outlet_label"].unique()):
    outlet_sig = sig_topics[sig_topics["outlet_label"] == outlet]
    over = outlet_sig[outlet_sig["diff_pp"] > 0].nlargest(5, "diff_pp")
    under = outlet_sig[outlet_sig["diff_pp"] < 0].nsmallest(5, "diff_pp")
    
    print(f"\n--- {outlet} ---")
    print(f"  Over-represented (top 5):")
    for _, row in over.iterrows():
        print(f"    {row['topic_label']:40s}  +{row['diff_pp']:5.1f}pp  (fold={row['fold_change']:.1f}x, p={row['p_corrected']:.2e})")
    print(f"  Under-represented (top 5):")
    for _, row in under.iterrows():
        print(f"    {row['topic_label']:40s}  {row['diff_pp']:5.1f}pp  (fold={row['fold_change']:.1f}x, p={row['p_corrected']:.2e})")

In [ ]:
# Volcano plots — one per outlet
for outlet in sorted(sig_topics["outlet_label"].unique()):
    fig = plot_chi_squared_volcano(
        chi_df,
        outlet=outlet,
        alpha=0.01,
        min_diff_pp=1.0,
        colors=THESIS_COLORS,
        save_path=fig_dir / f"h1_10_volcano_{outlet.lower().replace(' ', '_')}.pdf",
    )

### Cross-outlet topic significance summary

Which topics are significantly over-represented in 3+ alt outlets? This identifies the **common alt-media agenda** with statistical backing.

In [ ]:
# Topics significantly over-represented in 3+ alt outlets
over_rep = sig_topics[sig_topics["diff_pp"] > 0]
topic_outlet_count = over_rep.groupby("merged_topic").agg(
    topic_label=("topic_label", "first"),
    n_outlets=("outlet_label", "nunique"),
    outlets=("outlet_label", lambda x: ", ".join(sorted(x))),
    mean_diff_pp=("diff_pp", "mean"),
    max_diff_pp=("diff_pp", "max"),
).reset_index()

shared_agenda = topic_outlet_count[topic_outlet_count["n_outlets"] >= 3].sort_values(
    "mean_diff_pp", ascending=False
)

print("Topics significantly over-represented in 3+ alt outlets:")
print("=" * 90)
for _, row in shared_agenda.iterrows():
    print(f"  {row['topic_label']:40s}  outlets={row['n_outlets']}  "
          f"mean_diff=+{row['mean_diff_pp']:.1f}pp  max_diff=+{row['max_diff_pp']:.1f}pp")
    print(f"    -> {row['outlets']}")

---
## 3. Global Model Validation

Train a single global BERTopic on all 20,455 articles (the literature-standard approach),
then compare JSD rankings to our merged model.

**If rankings agree**: The merged-model approach is validated — it captures the same
agenda structure while preserving outlet-specific topics.

**If rankings diverge**: We need to understand why and report both.

> **Note**: This cell is computationally expensive (~10-15 min with GPU). 
> Skip if already computed — load from `outputs/global_v1/` instead.

In [ ]:
# Option A: Train global model (uncomment to run)
# from modeling import run_global_model, save_iteration
# global_result = run_global_model(PROJECT_ROOT, iteration_id="global_v1")
# save_iteration(global_result, OUTPUT_DIR)

# Option B: Load pre-computed global model
try:
    global_result = load_iteration(OUTPUT_DIR, "global_v1")
    print(f"Loaded global model: {global_result.n_topics} topics, "
          f"{len(global_result.merged_articles):,} articles")
    GLOBAL_AVAILABLE = True
except FileNotFoundError:
    print("Global model not yet computed. Uncomment Option A above and run.")
    print("Skipping global model comparison.")
    GLOBAL_AVAILABLE = False

In [ ]:
if GLOBAL_AVAILABLE:
    from scipy.stats import spearmanr
    
    # Compute H1 metrics for both models
    merged_metrics = compute_all_h1_metrics(v1.merged_articles)
    global_metrics = compute_all_h1_metrics(global_result.merged_articles)
    
    # Compare JSD rankings
    comparison = merged_metrics[["outlet_label", "jsd_vs_tagesschau"]].merge(
        global_metrics[["outlet_label", "jsd_vs_tagesschau"]],
        on="outlet_label",
        suffixes=("_merged", "_global"),
    )
    comparison = comparison[comparison["outlet_label"] != "Tagesschau"]
    
    # Spearman rank correlation of JSD values
    rho, pval = spearmanr(
        comparison["jsd_vs_tagesschau_merged"],
        comparison["jsd_vs_tagesschau_global"],
    )
    
    print("JSD Ranking Comparison: Merged Model vs Global Model")
    print("=" * 70)
    print(f"\n{'Outlet':<25} {'Merged JSD':>12} {'Rank':>6} {'Global JSD':>12} {'Rank':>6}")
    print("-" * 70)
    
    comp_sorted = comparison.sort_values("jsd_vs_tagesschau_merged", ascending=False)
    for i, (_, row) in enumerate(comp_sorted.iterrows()):
        merged_rank = comparison["jsd_vs_tagesschau_merged"].rank(ascending=False)[row.name]
        global_rank = comparison["jsd_vs_tagesschau_global"].rank(ascending=False)[row.name]
        print(f"{row['outlet_label']:<25} {row['jsd_vs_tagesschau_merged']:>12.4f} {merged_rank:>6.0f} "
              f"{row['jsd_vs_tagesschau_global']:>12.4f} {global_rank:>6.0f}")
    
    print(f"\nSpearman rank correlation: rho = {rho:.3f}, p = {pval:.4f}")
    print(f"Interpretation: {'RANKINGS AGREE' if rho > 0.7 else 'RANKINGS DIVERGE — investigate'}")
    
    # Also compare outlier rates
    print(f"\nOutlier rates:")
    print(f"  Merged model:  {v1.outlier_rates}")
    print(f"  Global model:  {global_result.outlier_rates}")
    print(f"\nTopics discovered:")
    print(f"  Merged model:  {v1.n_topics}")
    print(f"  Global model:  {global_result.n_topics}")
    
    # Save comparison
    comparison.to_csv(OUTPUT_DIR / "v1" / "global_vs_merged_comparison.csv", index=False)

---
## Summary

| Strengthening | What it adds | Key output |
|---------------|-------------|------------|
| Outlet clustering | Empirical validation of outlet categories | Dendrogram + pairwise JSD heatmap |
| Chi-squared tests | Statistical backing for topic divergence claims | Per-topic p-values + volcano plots |
| Global model | Validates merged-model approach against literature standard | JSD ranking comparison |

### Files produced
- `outputs/v1/pairwise_jsd.csv` — 7×7 pairwise JSD matrix
- `outputs/v1/chi_squared_topic_tests.csv` — all (outlet, topic) chi-squared results
- `outputs/v1/figures/h1_08_pairwise_jsd_heatmap.pdf`
- `outputs/v1/figures/h1_09_outlet_dendrogram.pdf`
- `outputs/v1/figures/h1_10_volcano_*.pdf` — per-outlet volcano plots